# 00 · Environment setup

Run this first on any machine. It checks:
1. Where we are (Colab / Vast.ai / local Mac)
2. Packages
3. API keys (HF token)
4. GPU / device
5. The model loads and generates
6. The three datasets are reachable

## 1. Detect environment

In [ ]:
import os, sys, platform

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")  # browser or VS Code Colab extension
IN_VAST = "VAST_CONTAINERLABEL" in os.environ or os.path.exists("/workspace")
ENV = "colab" if IN_COLAB else "vast" if IN_VAST else "local"
print(f"ENV={ENV} | Python {sys.version.split()[0]} | {platform.machine()}")

## 2. Install packages (Colab / Vast only — locally use `uv pip install -r requirements.txt`)

In [ ]:
if ENV in ("colab", "vast"):
    %pip install -q "transformers>=4.46" "peft>=0.13" "accelerate>=1.0" "datasets>=3.0" scikit-learn python-dotenv

## 3. API keys

- **Colab:** add `HF_TOKEN` in the 🔑 *Secrets* sidebar. If that fails (e.g. from VS Code), you'll be prompted to paste it.
- **Local / Vast:** copy `.env.example` → `.env` at repo root and fill it in.

In [ ]:
if ENV == "colab":
    try:  # Colab Secrets (may not work via the VS Code extension)
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        from getpass import getpass
        os.environ["HF_TOKEN"] = getpass("HF_TOKEN: ")
else:
    from dotenv import load_dotenv
    load_dotenv()  # searches upward for .env

print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

## 4. Device

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
    print("GPU:", torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print("bf16 supported:", torch.cuda.is_bf16_supported())  # False on T4 -> use fp16
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Apple Silicon (MPS) — fine for inference/debugging, not for training")
else:
    DEVICE = "cpu"
    print("CPU only")

DTYPE = torch.float16 if DEVICE in ("cuda", "mps") else torch.float32
print("torch", torch.__version__, "| DEVICE =", DEVICE, "| DTYPE =", DTYPE)

## 5. Model smoke test (Qwen2.5-1.5B-Instruct)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE).eval()
print("layers:", model.config.num_hidden_layers, "| hidden:", model.config.hidden_size)

msgs = [{"role": "user", "content": "Write a Python function that reverses a string."}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)["input_ids"].to(DEVICE)
with torch.no_grad():
    out = model.generate(ids, max_new_tokens=60, do_sample=False)
print(tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True))

Activation check: residual stream at the final prompt token, all layers (what the probes will read).

In [ ]:
with torch.no_grad():
    hs = model(ids, output_hidden_states=True).hidden_states
acts = torch.stack([h[0, -1] for h in hs])  # (n_layers+1, hidden)
print("activations:", tuple(acts.shape))

## 6. Datasets reachable

In [ ]:
from datasets import load_dataset

DATASETS = {
    "ultrachat (non-coding)": ("HuggingFaceH4/ultrachat_200k", "train_sft"),
    "code_alpaca (coding)":   ("sahil2801/CodeAlpaca-20k", "train"),
    "alpaca (FPR calib)":     ("tatsu-lab/alpaca", "train"),
}
for name, (repo, split) in DATASETS.items():
    ds = load_dataset(repo, split=f"{split}[:3]")
    print(f"✓ {name:24s} {repo}  cols={ds.column_names}")

✅ If every cell ran, the environment is ready.